# Stage 1 data pulls: Kalshi CPI, Cleveland Fed nowcast, 2-year Treasury

Run the cells top to bottom (Runtime > Run all works). Outputs go to `DATA_DIR`. With `MOUNT_DRIVE = True` they are saved in your Google Drive so they survive the Colab session ending.

No-look-ahead notes
- Kalshi `close_time` is a few minutes before the 8:30 ET release; snapshot from candles ending before it.
- A nowcast dated d is the value shown on d (~10:00 ET update): use dates strictly before release day.
- The realized print comes from the settled ladder (`print_flag == "exact"`); `interval` means a strike gap, check by hand.
- `no_report` marks Oct 2025 (no BLS release); drop it.

## 1. Settings

In [ ]:
MOUNT_DRIVE = True          # save outputs to Google Drive
SERIES = ["KXCPI", "KXCPICORE"]   # headline and core CPI m/m
LOOKBACK_DAYS = 7           # days of candles before each market closes
PERIOD = 60                 # candle length in minutes: 1, 60 or 1440
SINCE = "2022-03-01"        # skip events closing before this date
FRED_SERIES = ["DGS2"]
FRED_START = "2021-01-01"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/macro-news-pricing/data/raw"
else:
    DATA_DIR = "/content/data/raw"
!pip -q install pyarrow
print("Saving to", DATA_DIR)

## 2. Helpers

In [ ]:
import time
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RAW = Path(DATA_DIR)
RAW.mkdir(parents=True, exist_ok=True)

BROWSER_UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"
)


def session(min_interval: float = 0.12) -> requests.Session:
    """Session that retries on 429/5xx and spaces out requests (Kalshi basic tier ~10 req/s)."""
    s = requests.Session()
    retry = Retry(total=6, backoff_factor=1.0, status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent": BROWSER_UA, "Accept": "application/json,text/csv,*/*"})
    last = [0.0]
    orig = s.get

    def throttled_get(url, **kw):
        wait = min_interval - (time.monotonic() - last[0])
        if wait > 0:
            time.sleep(wait)
        last[0] = time.monotonic()
        kw.setdefault("timeout", 30)
        return orig(url, **kw)

    s.get = throttled_get  # type: ignore[method-assign]
    return s


def num(d: dict, key: str):
    """Read a numeric field that may be `key`, `key_fp` or `key_dollars` (Kalshi changed names)."""
    for k in (key, f"{key}_fp", f"{key}_dollars"):
        v = d.get(k)
        if v is not None and v != "":
            try:
                return float(v)
            except (TypeError, ValueError):
                return None
    return None


## 3. 2-year Treasury yield (quick test)
Tries FRED first, then the same 2-year par yield straight from Treasury.gov (FRED's DGS2 is Treasury's series), then asks you to upload a CSV. FRED often hangs for Colab's Google Cloud addresses, which is the timeout you may see.

In [ ]:
import io, requests

def fred_csv(sid, start):
    # plain client, short timeout, no retry storm
    r = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                     params={"id": sid, "cosd": start}, timeout=20,
                     headers={"User-Agent": "python-requests (research script)"})
    r.raise_for_status()
    d = pd.read_csv(io.StringIO(r.text), na_values=".")
    d.columns = ["date", sid]
    return d.assign(date=pd.to_datetime(d.date))

def treasury_2y(start):
    s = session(min_interval=0.5)
    out = []
    for yr in range(int(start[:4]), pd.Timestamp.today().year + 1):
        url = ("https://home.treasury.gov/resource-center/data-chart-center/interest-rates/"
               f"daily-treasury-rates.csv/{yr}/all")
        r = s.get(url, params={"type": "daily_treasury_yield_curve",
                               "field_tdr_date_value": yr, "page": "", "_format": "csv"}, timeout=30)
        r.raise_for_status()
        d = pd.read_csv(io.StringIO(r.text))
        out.append(pd.DataFrame({"date": pd.to_datetime(d["Date"]), "DGS2": d["2 Yr"]}))
    return pd.concat(out)

fred = None
for name, fn in (("FRED", lambda: fred_csv("DGS2", FRED_START)), ("Treasury.gov", lambda: treasury_2y(FRED_START))):
    try:
        fred = fn(); print("2-year yield from", name); break
    except Exception as e:
        print(f"{name} failed: {type(e).__name__}: {str(e)[:120]}")

if fred is None:
    print("Both blocked. Download https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS2 "
          "in your browser, then pick the file below.")
    from google.colab import files
    up = files.upload()
    fred = pd.read_csv(io.BytesIO(next(iter(up.values()))), na_values=".")
    fred.columns = ["date", "DGS2"]
    fred["date"] = pd.to_datetime(fred.date)

fred = (fred[fred.date >= FRED_START].dropna().drop_duplicates("date")
        .sort_values("date").reset_index(drop=True))
fred.to_parquet(RAW / "fred_daily.parquet", index=False)
print(f"{len(fred)} rows, {fred.date.min():%Y-%m-%d} to {fred.date.max():%Y-%m-%d}")
fred.tail()


## 4. Kalshi: CPI markets and hourly candles (about 6 minutes)
If it stops partway, rerun this cell: candles are cached, so it resumes.

In [ ]:
import argparse
import json
import re
from datetime import datetime, timedelta, timezone

import pandas as pd


BASE = "https://api.elections.kalshi.com/trade-api/v2"
NO_REPORT = {"25OCT"}  # BLS skipped Oct 2025 CPI (shutdown); markets settled on fallback
STRIKE_RE = re.compile(r"-T(N?)(\d+(?:\.\d+)?)$")
CACHE = RAW / "kalshi_json"
CACHE.mkdir(exist_ok=True)


def paginate(s, url, params, key):
    out, cursor = [], None
    while True:
        p = dict(params, limit=1000, **({"cursor": cursor} if cursor else {}))
        r = s.get(url, params=p)
        r.raise_for_status()
        j = r.json()
        out += j.get(key, [])
        cursor = j.get("cursor")
        if not cursor:
            return out


def parse_strike(m):
    if m.get("floor_strike") is not None:
        return float(m["floor_strike"])
    mt = STRIKE_RE.search(m["ticker"])  # legacy markets (2021-22) have no floor_strike
    if mt:
        return (-1 if mt.group(1) else 1) * float(mt.group(2))
    return None


def get_markets(s, series):
    hist = paginate(s, f"{BASE}/historical/markets", {"series_ticker": series}, "markets")
    live = paginate(s, f"{BASE}/markets", {"series_ticker": series, "status": "settled"}, "markets")
    rows = {}
    for src, ms in (("historical", hist), ("live", live)):
        for m in ms:
            rows.setdefault(m["ticker"], {
                "ticker": m["ticker"],
                "event_ticker": m["event_ticker"],
                "series": series,
                "source": src,
                "strike": parse_strike(m),
                "strike_type": m.get("strike_type") or "greater",  # legacy rules: "more than X%"
                "subtitle": m.get("yes_sub_title") or m.get("subtitle"),
                "open_time": m.get("open_time"),
                "close_time": m.get("close_time"),
                "result": m.get("result"),
                "expiration_value": m.get("expiration_value"),
                "volume": num(m, "volume"),
                "open_interest": num(m, "open_interest"),
                "last_price": num(m, "last_price"),
                "rules_primary": m.get("rules_primary"),
            })
    df = pd.DataFrame(rows.values())
    for c in ("open_time", "close_time"):
        df[c] = pd.to_datetime(df[c], utc=True, format="ISO8601")
    return df


def realized_print(g):
    """Print = (highest strike settling yes) + 0.1, if the next strike up exists and settled no."""
    g = g.sort_values("strike")
    yes, no = g.loc[g.result == "yes", "strike"], g.loc[g.result == "no", "strike"]
    consistent = yes.empty or no.empty or yes.max() < no.min()
    if yes.empty:
        return pd.Series({"print": None, "print_lo": None, "print_hi": no.min(),
                          "print_flag": "below_ladder", "consistent": consistent})
    if no.empty:
        return pd.Series({"print": None, "print_lo": round(yes.max() + 0.1, 1), "print_hi": None,
                          "print_flag": "above_ladder", "consistent": consistent})
    lo, hi = round(yes.max() + 0.1, 1), round(no.min(), 1)
    exact = abs(lo - hi) < 1e-9
    return pd.Series({"print": lo if exact else None, "print_lo": lo, "print_hi": hi,
                      "print_flag": "exact" if exact else "interval", "consistent": consistent})


def summarize_events(mk):
    g = mk.groupby("event_ticker")
    ev = g.agg(series=("series", "first"), n_strikes=("ticker", "size"),
               min_strike=("strike", "min"), max_strike=("strike", "max"),
               volume=("volume", "sum"), open_time=("open_time", "min"),
               close_time=("close_time", "max")).reset_index()
    ev = ev.merge(g.apply(realized_print, include_groups=False).reset_index(), on="event_ticker")
    ev["release_date_et"] = ev.close_time.dt.tz_convert("America/New_York").dt.date
    ev["month_code"] = ev.event_ticker.str.split("-").str[1]
    ev["no_report"] = ev.month_code.isin(NO_REPORT)
    ev["close_et"] = ev.close_time.dt.tz_convert("America/New_York").dt.strftime("%H:%M")
    return ev.sort_values("close_time").reset_index(drop=True)


def candle_rows(ticker, cs):
    rows = []
    for c in cs:
        row = {"ticker": ticker, "end_ts": c["end_period_ts"],
               "volume": num(c, "volume"), "open_interest": num(c, "open_interest")}
        for side in ("yes_bid", "yes_ask", "price"):
            for f in ("open", "high", "low", "close") + (("mean",) if side == "price" else ()):
                row[f"{side}_{f}"] = num(c.get(side) or {}, f)
        rows.append(row)
    return rows


def get_candles(s, m, lookback_days, period):
    path = CACHE / f"{m.ticker}_p{period}_lb{lookback_days}.json"
    if path.exists():
        return json.loads(path.read_text())
    end = m.close_time + timedelta(hours=1)
    start = m.close_time - timedelta(days=lookback_days)
    params = {"start_ts": int(start.timestamp()), "end_ts": int(end.timestamp()),
              "period_interval": period}
    if m.source == "historical":
        url = f"{BASE}/historical/markets/{m.ticker}/candlesticks"
    else:
        url = f"{BASE}/series/{m.event_ticker.split('-')[0]}/markets/{m.ticker}/candlesticks"
    r = s.get(url, params=params)
    r.raise_for_status()
    cs = r.json().get("candlesticks", [])
    path.write_text(json.dumps(cs))
    return cs

s = session()
mk = pd.concat([get_markets(s, x) for x in SERIES], ignore_index=True)
mk.to_parquet(RAW / "kalshi_markets.parquet", index=False)
ev = summarize_events(mk)
ev.to_parquet(RAW / "kalshi_events.parquet", index=False)

since = pd.Timestamp(SINCE, tz="UTC")
todo = mk[(mk.close_time >= since) & (mk.result.isin(["yes", "no"]))]
print(f"{len(mk)} markets / {len(ev)} events; fetching candles for {len(todo)} markets")
rows = []
for i, m in enumerate(todo.itertuples(), 1):
    try:
        rows += candle_rows(m.ticker, get_candles(s, m, LOOKBACK_DAYS, PERIOD))
    except Exception as e:
        print(f"  candle fail {m.ticker}: {e}")
    if i % 50 == 0:
        print(f"  {i}/{len(todo)}")
cd = pd.DataFrame(rows)
cd["end_time"] = pd.to_datetime(cd.end_ts, unit="s", utc=True)
cd.to_parquet(RAW / "kalshi_candles.parquet", index=False)
print(f"candles: {len(cd)} rows")
pd.set_option("display.max_rows", 200)
ev[["event_ticker", "release_date_et", "close_et", "n_strikes", "min_strike", "max_strike",
    "volume", "print", "print_flag", "consistent", "no_report"]]


## 5. Cleveland Fed nowcast paths
If the download is blocked: open the URL printed by the error in your browser, save the page as `nowcast_month.json`, set `NOWCAST_FILE = "upload"` below and rerun; Colab will ask you to pick the file.

In [ ]:
NOWCAST_FILE = None   # or "upload"

import argparse
import json
from datetime import date

import pandas as pd


NOWCAST_URL = ("https://www.clevelandfed.org/-/media/files/webcharts/inflationnowcasting/"
               "nowcast_month.json?sc_lang=en")
MEASURES = {"CPI Inflation": "cpi", "Core CPI Inflation": "core_cpi",
            "PCE Inflation": "pce", "Core PCE Inflation": "core_pce"}


def load(file):
    if file:
        return json.loads(open(file, encoding="utf-8-sig").read())
    s = session(min_interval=0)
    s.headers.update({"Referer": "https://www.clevelandfed.org/indicators-and-data/inflation-nowcasting"})
    r = s.get(NOWCAST_URL, timeout=120)
    r.raise_for_status()
    (RAW / "nowcast_month.json").write_bytes(r.content)  # keep the raw vintage
    return r.json()


def to_date(mmdd: str, ty: int, tm: int) -> date:
    """Chart labels are MM/DD with no year; paths start in the target month and run forward."""
    mo, dy = (int(x) for x in mmdd.split("/"))
    yr = ty + 1 if mo < tm and tm - mo > 6 else ty  # e.g. Dec target, Jan/Feb dates
    return date(yr, mo, dy)


def parse(charts):
    rows = []
    for ch in charts:
        ty, tm = (int(x) for x in ch["chart"]["subcaption"].split("-"))
        for ds in ch["dataset"]:
            name = ds["seriesname"]
            kind = "actual" if name.startswith("Actual ") else "nowcast"
            measure = MEASURES.get(name.removeprefix("Actual "))
            if measure is None:
                continue
            for pt in ds["data"]:
                v = pt.get("value")
                if v in (None, ""):
                    continue
                parts = pt.get("tooltext", "").split("{br}")  # "CPI Inflation{br}08/13{br}0.30{br}"
                if len(parts) < 2 or "/" not in parts[1]:
                    continue
                rows.append({"target_month": f"{ty}-{tm:02d}", "date": to_date(parts[1].strip(), ty, tm),
                             "measure": measure, "kind": kind, "value": float(v)})
    df = pd.DataFrame(rows).drop_duplicates()
    df["date"] = pd.to_datetime(df.date)
    return df.sort_values(["target_month", "measure", "date"]).reset_index(drop=True)

if NOWCAST_FILE == "upload":
    from google.colab import files
    up = files.upload()
    charts = json.loads(next(iter(up.values())).decode("utf-8-sig"))
    (RAW / "nowcast_month.json").write_text(json.dumps(charts))
else:
    try:
        charts = load(None)
    except Exception as e:
        raise SystemExit(f"Download failed ({e}). Open {NOWCAST_URL} in a browser, save it, "
                         "set NOWCAST_FILE = 'upload' and rerun this cell.")
nw = parse(charts)
nw.to_parquet(RAW / "nowcast_long.parquet", index=False)
nc = nw[(nw.kind == "nowcast") & nw.measure.isin(["cpi", "core_cpi"])]
wide = nc.pivot_table(index=["target_month", "date"], columns="measure", values="value").reset_index()
wide.to_parquet(RAW / "nowcast_cpi.parquet", index=False)
nw[nw.kind == "actual"].to_parquet(RAW / "nowcast_actuals.parquet", index=False)
print(f"{nw.target_month.nunique()} target months, {len(nw)} rows; generated {charts[-1]['chart'].get('_comment')}")
nc[nc.measure == "cpi"].groupby("target_month").date.agg(["min", "max", "size"]).tail(15)


## 6. Check what was saved

In [ ]:
for p in sorted(RAW.glob("*.parquet")):
    print(f"{p.name:28s} {len(pd.read_parquet(p)):>8,} rows")